# IA 音乐作品分析平台 — 开发技术报告

> 大一 Python 课程大作业。记录从零到一的完整开发过程，包括爬虫、架构、分类器、前端。
> 按时间顺序，保留错误尝试与反思。

## 目录

1. [项目需求与选题](#1-项目需求与选题)
2. [架构设计](#2-架构设计)
3. [B站爬虫](#3-b站爬虫)
4. [内容分类：QLoRA 微调](#4-内容分类qlora-微调)
5. [内容分类：正则表达式迭代](#5-内容分类正则表达式迭代)
6. [Flask 项目骨架与前端](#6-flask-项目骨架与前端)
7. [当前状态与后续计划](#7-当前状态与后续计划)

## 1. 项目需求与选题

### 1.1 课程要求

实现五大模块：数据文件读取 → 数据预处理 → 数据分析 → 数据可视化 → 交互式分析（AI问答）。
数据集不少于 1000 行、10 语义列，支持上传不同格式文件，前端 UI + 可视化。

### 1.2 选题

**B站 IA 虚拟歌姬音乐作品播放量与传播分析**。

核心问题：
- IA 音乐作品随时间推移的热度趋势——"IA 过气了吗？"
- 音游（PJSK/BangDream/Phigros 等）对 IA 热度的贡献 vs VOCALOID 社区内部的热度
- 原创 vs 翻唱 vs 搬运的表现差异
- 作者/P主 影响力排行与活跃度

### 1.3 技术选型理由

| 组件 | 选择 | 原因 |
|------|------|------|
| 后端框架 | Flask | 轻量、Python 原生、课程已涉及 |
| 数据处理 | pandas | 课堂核心内容 |
| 前端图表 | ECharts | 交互性强、CDN 引入、无需 npm |
| CSS | Bootstrap 5 | 响应式、Admin Dashboard 风格 |
| AI 分类 | Qwen2.5-1.5B + QLoRA | 本地推理、Mac 可跑 |
| AI 问答 | DeepSeek V4 Flash + Function Calling | 性价比高、中文强 |

## 2. 架构设计

### 2.1 初始设计

从需求出发，画出了五大模块的数据流：

```
文件上传 → 预处理 → 分析引擎 → 可视化 → AI问答
```

预处理模块需要产出两个粒度的数据：
- **视频粒度** (`df_clean`)：分析 UP主 行为
- **歌曲粒度** (`df_songs`)：分析歌曲热度演化（过气分析基础）

### 2.2 关键设计决策

**1. 为什么需要歌曲粒度？**

4000+ 条视频记录中，同一首歌可能被 20 个 UP主 分别上传。"六兆年と一夜物語 这首歌的火热程度是否在下降？"这类问题必须在歌曲粒度上回答。

聚合键：`(song_name, original_creator)`——同一首歌 + 同一 P主 = 同一条歌曲记录。

**2. P主 粒度需要独立 DataFrame 吗？**

不需要。从 `df_songs` 对 `original_creator` 做 `groupby` 就能得到，毫秒级。三个粒度的关系：

```
视频 (4000+条) ──聚合──→ 歌曲 (几百条) ──聚合──→ P主 (几十条)
   ↑                      ↑                    ↑
 df_clean              df_songs          按需 groupby
```

**3. 音游视频应该被排除吗？**

不。正确做法是**分层对比**：全量趋势 vs VOCALOID核心(排除音游) vs 音游趋势。三条线放在同一张图里，才能判断热度变化到底是"VOCALOID社区在萎缩"还是"音游换了一波游戏"。这也是后期设计 `content_type` 分类 + `is_game` 标记的原因。

**4. 页面布局：Admin Dashboard**

放弃传统的多页跳转，改用 Admin Dashboard 风格：左侧固定导航 + 顶部状态卡片 + 主内容区动态切换。三个子页面（数据加载/分析/问答）共享布局，切换不丢失数据状态。

## 3. B站爬虫

### 3.1 API 选型

使用 `bilibili-api-python` 库，两个核心 API：
- `search_by_type(keyword, video_zone_type=30)` → 搜索 VOCALOID·UTAU 分区
- `Video(bvid).get_info()` → 获取播放量/硬币/点赞等完整统计

### 3.2 字段缺口发现

搜索 API 返回 12/20 个字段，缺少 `coin`、`share`、`like`、`category`。需要逐视频调用详情 API 补全。这是爬取时间的主要开销。

### 3.3 错误尝试

**错误 1：category 来源选错**

最初设计从详情 API 的 `tname` 取分区名。实际测试发现 `tname` 经常为空（如 tid=30 的 VOCALOID·UTAU 视频返回空字符串）。改为从搜索 API 的 `typename` 取值，更可靠。

**错误 2：翻页无限循环**

B站搜索翻到极限后不返回空，而是循环返回已爬数据。每页新增为 0 但 `items` 不为空，导致永不终止。修复：新增为 0 时 `break`。

**错误 3：多P视频去重遗漏**

搜索阶段用 `(bvid, 1)` 去重，多P视频（如 page=7）不在集合中会重复爬取。修复：搜索阶段用唯一 `bvid` 去重，详情阶段用 `(bvid, page)`。

**错误 4：TARGET_COUNT 未限制详情**

搜索达到 100 条后停止，但详情阶段处理了全部 756 条。修复：详情阶段只处理 `new_videos[:TARGET_COUNT]`。

### 3.4 爬虫最终产出

- 数据：4,128 行 × 20 列
- 时间：~60 分钟（搜索 4min + 详情 50min）
- 无风控触发（1~3s 随机间隔）

## 4. 内容分类：QLoRA 微调

### 4.1 动机

爬虫混入了大量噪音——B站上"IA"不只是虚拟歌姬，也是飞机型号（IA58）、音响型号（LM-519IA）、游戏的 AI 笔误。纯正则很容易被"IA"这个词欺骗。（早知道做Miku了）

### 4.2 技术方案

- 模型：Qwen2.5-1.5B-Instruct
- 微调：QLoRA (4-bit, r=8, alpha=16)
- 框架：unsloth (MLX 后端，Mac M系列可用)
- 数据：260 条人工标注 → 后来扩展到 1,989 条

### 4.3 训练过程踩坑

**坑 1：HuggingFace 被墙**

直接下载模型超时。解决：`HF_ENDPOINT=https://hf-mirror.com` 国内镜像。

**坑 2：预量化模型不兼容 MLX**

`unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit` 是 bitsandbytes 格式，Mac MLX 后端不认。
解决：换 `Qwen/Qwen2.5-1.5B-Instruct`（基座），让 MLX 自行量化。

**坑 3：Python 3.14 + dill 不兼容**

`datasets.Dataset.from_list()` 触发 pickle 错误。
解决：绕过 datasets，手写 PyTorch Dataset。

**坑 4：`transformers.Trainer` 不兼容 MLX 模型**

MLX 模型不是 `nn.Module`，`Trainer` 拒绝工作。
解决：换 unsloth 内置的 `MLXTrainer` + `MLXTrainingConfig`。

**坑 5：`save_pretrained_gguf` 需要 cmake**

系统没有 cmake，无法编译 llama.cpp。
解决：改用 `save_lora_adapters()` 保存 safetensors，评测用 `mlx_lm.load(adapter_path=...)`。

**坑 6：`class_weights` 算出来但从未使用**

MLXTrainer 不支持 `class_weight` 参数。
解决：改用**上采样**——少数类复制达到多数类数量。

**坑 7：60 steps 太少**

MLXTrainer 默认 `max_steps=60`。223 条训练样本 + batch=2 需要 ~112 步/epoch。
解决：设置 `max_steps=-1, num_train_epochs=3`。

### 4.4 训练结果

准确率 70%（37 条测试集）。vocaloid_original 类样本 72 条远超其他类（15-51 条），模型偏向多数类。即使上采样后，60 steps 也不足以学习区分。

### 4.5 反思

QLoRA 在 Mac 上的生态还不够成熟——MLX/Python3.14/datasets 的兼容性花了很多时间。核心矛盾是：**20% 的边界案例决定了分类质量，但少量标注的 QLoRA 模型很难学到这些边界**。正则虽然覆盖率有限，但迭代快、可解释、不需要 GPU 等待。最终决定：正则为主，QLoRA 留到后期作为可选增强。

## 5. 内容分类：正则表达式迭代

### 5.1 从五分类到三分类

最初设计了 5 种 content_type：`game_cover`、`vocaloid_original`、`vocaloid_cover`、`irrelevant`、`other`。

经过对分析需求的审视，发现：
- 过气分析只需要"是不是IA音乐"
- 音游/翻唱的区别通过布尔标记即可
- `other` 和 `irrelevant` 的边界本身模糊

**简化方案（v2）**：

```
三分类: ia_music | ia_related | irrelevant
独立标记: is_game (bool) + is_cover (bool)
```

分析需求通过标记组合实现：
- 过气分析 → `ia_music`
- 音游分层 → `ia_music ∩ is_game=True`
- 原创 vs 翻唱 → `ia_music ∩ is_cover`

### 5.2 迭代过程

**v1（基线）**：仅基于 title 关键词匹配。准确率 ~67%。

**v1.1**：发现 tags 里的翻唱关键词没被检查——步骤 4 只看 title，步骤 1（音游）却同时检查 title+tags。修复一致性。准确率 ↑。

**v1.2**：发现 `is_reupload` 死代码——正则分类在标题解析之前调用，`is_reupload` 永远是 False。调换步骤顺序。

**v2（数据驱动重构）**：扔掉旧正则，基于 260 条标注数据的统计分析重新设计规则。核心发现：

| 信号 | 数据支持 |
|------|---------|
| `category == VOCALOID·UTAU` | 93% 的 vocaloid_original 在此分区 |
| tags 无 IA 且无 VOCALOID | 83% 的 irrelevant 无此标签 |
| tags 含 IA 或 VOCALOID | 97%+ 的音乐类有此标签 |
| Cover/翻唱关键词 | 32% 的 vocaloid_cover 标题含此 |

准确率跳升到 80%。

**v2.1-v2.n（迭代优化）**：

用户对分类结果进行人工审查，将错误分类的视频手工筛选到对应 CSV 中。每次审查暴露一类规律性问题。

典型迭代案例：
- **发现**：大量 `irrelevant → ia_music` 误判来自"漫剧"等 tag 未被检测 → **修复**：`IRRELEVANT_SIGNALS` 加入标题关键词
- **发现**：tags 里"漫剧"标注但标题无"漫剧"→ `IRRELEVANT_SIGNALS` 只查 title 不查 tags → **修复**：移入 `HIGH_PRIORITY_IRRELEVANT`（title+tags 双查）
- **发现**：`itemasylum`（无空格）未命中 `Item Asylum` → **修复**：加无空格变体
- **发现**：标签"激励计划""kpop"在无关视频中频繁出现 → **修复**：加入关键词列表
- **发现**：`only_ia_signal`（IA 标签孤立的视频）大量误判 → **修复**：标记为可疑而非自信分类

### 5.3 最终评估

| 指标 | 值 |
|------|-----|
| 标注集 | 1,989 条 |
| 准确率 | **94.3%** |
| ia_music 召回率 | **99%**（1,471/1,480） |
| 可疑率 | 15.9%（316 条标记待 LLM） |

剩余 5.7% 误判主要是 `ia_related → ia_music`（73 条），属于"看起来像歌但不是歌"的语义问题——正则无法解决的边界。这些被标记为 `suspicious=True`，等待 LLM 二次确认。

### 5.4 正则分类的架构

```
classify(title, tags, category)
  │
  ├─ 提取信号: has_vocaloid_tag, has_singer_tag, has_ia_exact,
  │            only_ia_signal, strong_title_music, weird_title
  │
  ├─ 步骤1: HIGH_PRIORITY → title/tags → irrelevant（无视一切）
  ├─ 步骤2: 完全没信号 + 非音乐分区 → irrelevant
  ├─ 步骤3: IRRELEVANT_SIGNALS → title → irrelevant
  ├─ 步骤4: NON_SONG_KEYWORDS → title → ia_related
  ├─ 步骤5: music_signal + 无【IA】→ ia_music（分情况标记可疑）
  ├─ 步骤6: 高置信度 → ia_music
  └─ 步骤7: category 兜底 → 标记可疑
  │
  └─ 输出: content_type, is_game, is_cover, rule, suspicious, suspicious_reason
```

### 5.5 为什么正则就够了

1. **速度**：正则毫秒级，4,128 行秒级完成；LLM 需要 ~1s/条
2. **可解释**：每条分类都知道命中了哪条规则
3. **可迭代**：发现误判 → 加一个关键词 → 立即修复，不需要重新训练
4. **覆盖率**：94.3% 准确率 + 99% 音乐召回，正则处理了绝大多数
5. **兜底**：15.9% 的可疑标记交给 LLM 做最后一道防线

## 6. Flask 项目骨架与前端

### 6.1 Flask 骨架

采用模块化设计，每个功能独立成文件：

```
modules/
├── file_reader.py         # 模块1：上传/预览/格式兼容
├── preprocessor.py        # 模块2：清洗管道编排
├── content_classifier.py  # 模块2子：正则分类器
├── analyzer.py            # 模块3：分析引擎（待实现）
├── visualizer.py          # 模块4：ECharts JSON（待实现）
└── ai_assistant.py        # 模块5：AI 问答（待实现）
```

`app.py` 负责路由注册和数据生命周期管理。遇到的一个问题是 Flask `g` 对象在请求间不持久——上传的数据在下一次 API 请求中丢失。解决：用服务端字典 `_data_store` 存储 DataFrame，session 仅存一个 `data_id` 索引。

### 6.2 前端模板

采用 Bootstrap 5 + ECharts 5，全部 CDN 引入，无需 npm。Admin Dashboard 布局：

- 左侧深色侧栏（220px 固定） → 品牌 Logo + 导航链接 + 数据状态指示器
- 顶部状态卡片 → 4 个指标（总记录数/字段数/时间范围/歌曲数），数据加载后显示
- 主内容区 → 三个子页面通过侧栏切换

页面设计：
- `index.html`：拖拽上传区 + 数据预览表格 + "前往分析"按钮
- `analysis.html`：左侧操作面板 + 右侧 ECharts 渲染区
- `chat.html`：对话界面 + 预设问题快捷按钮

### 6.3 文件上传模块

`file_reader.py` 支持 CSV/Excel/JSON 三种格式，通过扩展名自动识别。

其中 `detect_format()`、`read_csv()`、`read_excel()`、`read_json()` 四个函数由学生编写（约 20 行），作为编码练习。`get_preview()` 和 Flask 路由处理由 AI 辅助完成。

## 7. QLoRA 微调突破

### 7.1 转折点：Prompt 格式

之前的 QLoRA 评测显示 70% 准确率，一度决定放弃。后来发现**评测用的 prompt 格式与训练格式不一致**：

```
训练格式 (Alpaca):              错误评测格式:
### Instruction:                分类: ia_music/ia_related/irrelevant
<SYSTEM_PROMPT>                 标题: xxx
### Input:                      标签: xxx
标题: xxx                       分区: xxx
标签: xxx                       标签:    ← 模型不认识这个结尾
分区: xxx
### Response:
```

**根因**：QLoRA 微调让模型学会了"看到 `### Response:\n` 后输出标签"这个模式。评测时换了 prompt 格式，模型就输出 tags 而非标签——看起来像复述输入，实则是格式不匹配。

### 7.2 修正后结果

用正确 Alpaca 格式重新评测 1,989 条标注：

| 指标 | 正则 | QLoRA |
|------|:---:|:---:|
| 总准确率 | 94.3% | **98.2%** |
| ia_music 召回 | 99% | 99% |
| ia_related 召回 | 55% | **93%** |
| irrelevant 召回 | 91% | 96% |

QLoRA 在正则最弱的 `ia_related` 类上实现了质的飞跃（55% → 93%）。1.5B 参数 + 1,989 条标注足以完成这个三分类任务。

### 7.3 关键教训

**Prompt 格式一致性是 QLoRA 微调的生命线。** 训练时的 `### Response:\n` 结尾必须在推理时原样复制。任何改动——改标点、换措辞、调整结构——都会导致模型输出错乱。这个坑耽误了数天时间，直到逐条检查模型原始输出才发现格式不匹配。

## 8. 正则 + QLoRA 联合管道

### 8.1 设计思路

正则秒级处理 84% 的自信案例，QLoRA 处理剩余 16% 的可疑案例。两者互补而非替代。

### 8.2 保守策略：音乐纯净度优先

过气分析需要 `ia_music` 绝对纯净。因此制定保守规则：

> 正则和 QLoRA 在 music vs related 之间有分歧 → 归入 `ia_related`

### 8.3 联合管道结果

| 指标 | 纯正则 | 联合管道 |
|------|:---:|:---:|
| 总准确率 | 94.3% | **96.3%** |
| ia_music 召回 | 99% | 99% |
| ia_related 召回 | 37% | **66%** |
| irrelevant 召回 | 91% | 98% |
| Qwen 误改 | - | 11条 |

保守策略将 Qwen 误改从 55 条降到 11 条。联合管道准确率达 96.3%，ia_music 保持零污染。

### 8.4 最终分类方案

```
默认: 纯正则 (94.3%, 秒级)
可选: 正则 + QLoRA 联合 (96.3%, ~5min)
可选: 纯 QLoRA (98.2%, ~30min)
```

用户可在 Flask 前端选择清洗方案。QLoRA 本地免费，DeepSeek 需 API Key。

## 9. 当前状态与后续计划

### 9.1 已完成（~3,000 行代码）

| 模块 | 状态 | 行数 |
|------|:--:|:--:|
| 爬虫 | ✅ | 392 |
| Flask 骨架 | ✅ | 145 |
| 前端模板 | ✅ | ~350 |
| file_reader | ✅ | 210 |
| content_classifier (正则) | ✅ | 360 |
| llm_verifier (QLoRA/DeepSeek双后端) | ✅ | 220 |
| preprocessor (去重+分类+LLM验证) | ✅ | 137 |
| train_classifier (QLoRA训练) | ✅ | 355 |
| test_title_parser (标题解析原型) | ✅ | 200 |

### 9.2 待实现

| 模块 | 内容 | 预计行数 |
|------|------|:--:|
| `parse_titles()` | 标题解析 | ~100 |
| `general_clean()` + `add_features()` | 缺失值/日期/衍生特征 | ~70 |
| `aggregate_to_songs()` | 歌曲粒度聚合 | ~50 |
| `analyzer.py` | 21 个分析函数 | ~250 |
| `visualizer.py` | 12 个 ECharts JSON 函数 | ~200 |
| `ai_assistant.py` | DeepSeek Function Calling | ~150 |

### 9.3 关键经验

1. **QLoRA 训练和推理的 prompt 格式必须完全一致**——换个结尾符号模型就输出乱码。
2. **正则和 LLM 互补而非替代**——叠加需保守策略，否则 LLM 会误改正则正确结果。
3. **"有争议就归入低风险类"**——宁可 ia_music 小但干净。
4. **标注数据比算法更重要**——1,989 条标注是 96.3% 的基础。
5. **Python 3.14 兼容性**——datasets/dill/pickle 连环崩溃。</cell id="cell-8">
